# JS06 (ANN) - Praktikum 4: Perbandingan Annoy vs FAISS vs HNSW
**Mata Kuliah:** Pembelajaran Mesin  
**Nama:** Aryaputra | **NIM:** 244107020127  

---
### Deskripsi
Membandingkan ketiga model ANN (Annoy, FAISS Flat, dan HNSW) pada dataset berukuran besar untuk melihat build time dan query time.


In [1]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

n_data = 100_000
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# 1. Annoy
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')
start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)
build_t_ann = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k)
query_t_ann = time.time() - start
print(f"Build time: {build_t_ann:.3f} s | Query time: {query_t_ann*1000:.3f} ms")

# 2. FAISS Flat
print("\n=== FAISS Flat ===")
faiss_index = faiss.IndexFlatL2(dim)
start = time.time()
faiss_index.add(X)
build_t_faiss = time.time() - start

start = time.time()
dist_f, idx_f = faiss_index.search(query, k)
query_t_faiss = time.time() - start
print(f"Build time: {build_t_faiss:.3f} s | Query time: {query_t_faiss*1000:.3f} ms")

# 3. HNSW
print("\n=== HNSW ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)
start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_t_hnsw = time.time() - start

hnsw_index.set_ef(50)
start = time.time()
labels, dists = hnsw_index.knn_query(query, k=k)
query_t_hnsw = time.time() - start
print(f"Build time: {build_t_hnsw:.3f} s | Query time: {query_t_hnsw*1000:.3f} ms")


=== Annoy ===
Build time: 0.947 s | Query time: 0.187 ms

=== FAISS Flat ===
Build time: 0.000 s | Query time: 15.994 ms

=== HNSW ===
Build time: 0.000 s | Query time: 14.791 ms
